# Candidate Features H3 - Silver Layer

Generates expansion candidate locations with urbanicity-based variable K-ring trade areas.

**Key Features:**
- Uses pre-computed CARTO urbanicity column
- Variable K-ring sizes based on urbanicity (urban=2, suburban=3, rural=8)
- Aggregates demographics within K-ring trade areas
- Filters to top 25% candidates

**Inputs:**
- `{carto_table}` - CARTO Marketplace H3 features (configurable via widget)
- `{catalog}.{bronze_schema}.census_states` - MA boundary

**Output:**
- `{catalog}.{silver_schema}.expansion_candidates_h3`

**Note:** Exclusion of existing LCE store trade areas happens downstream in `expansion_prediction.ipynb`

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, explode, lit, when, udf
from pyspark.sql.types import IntegerType, ArrayType, StringType
import yaml

dbutils.widgets.text("catalog", "jdub_demo_aws")
dbutils.widgets.text("bronze_schema", "geo_bronze")
dbutils.widgets.text("silver_schema", "geo_silver")
dbutils.widgets.text("carto_table", "carto_spatial_features_usa_h3_res_8.carto.derived_spatialfeatures_usa_h3res8_v1_yearly_v3")
dbutils.widgets.text("config_path", "/Workspace/resources/configs/h3_features_config.yml")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
carto_table = dbutils.widgets.get("carto_table")
config_path = dbutils.widgets.get("config_path")

# Load config
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

H3_RESOLUTION = config['h3_grid']['resolution']

# Output table
output_table = f"{catalog}.{silver_schema}.expansion_candidates_h3"

print(f"CARTO source: {carto_table}")
print(f"H3 resolution: {H3_RESOLUTION}")
print(f"Output: {output_table}")

In [ ]:
# MAGIC %md
# MAGIC ## K-Ring Configuration Based on Urbanicity

In [ ]:
# K-ring sizes based on urbanicity
# Higher density = smaller trade area, lower density = larger trade area
URBANICITY_KRING_MAP = {
    "Very_High_density_urban": 2,  # ~0.87 mile radius
    "High_density_urban": 2,
    "Medium_density_urban": 3,     # ~1.45 mile radius  
    "Low_density_urban": 3,
    "Rural": 8                     # ~4.0 mile radius
}

# Default K-ring for unknown urbanicity
DEFAULT_KRING = 3

print("Urbanicity K-Ring Configuration:")
for urbanicity, k in URBANICITY_KRING_MAP.items():
    print(f"  {urbanicity}: K={k}")

In [ ]:
# MAGIC %md
# MAGIC ## Load CARTO Data and Filter to Massachusetts

In [ ]:
# Load Massachusetts boundary
ma_boundary = spark.table(f"{catalog}.{bronze_schema}.census_states").filter(
    (col("state_abbr") == "MA") | (col("state_fips") == "25")
)

print(f"Loaded Massachusetts boundary")

# Generate H3 grid covering Massachusetts
ma_h3_cells = ma_boundary.select(
    explode(expr("h3_coverash3string(ST_AsBinary(geometry), 5)")).alias("coarse_h3")
).select(
    explode(expr(f"h3_tochildren(coarse_h3, {H3_RESOLUTION})")).alias("h3_cell_id")
).distinct()

print(f"Generated {ma_h3_cells.count()} H3 cells covering Massachusetts")

In [ ]:
# Load CARTO features
carto_features = spark.table(carto_table)

# Check for urbanicity column (may be named differently in CARTO)
urbanicity_col = None
for col_name in ['urbanicity', 'urban_rural', 'urbanicity_category', 'urban_type']:
    if col_name in carto_features.columns:
        urbanicity_col = col_name
        break

if urbanicity_col:
    print(f"Found urbanicity column: {urbanicity_col}")
else:
    print("No urbanicity column found - will use population density to derive it")

# Filter CARTO to Massachusetts H3 cells
carto_ma = carto_features.join(
    ma_h3_cells,
    carto_features["h3"] == ma_h3_cells["h3_cell_id"],
    "inner"
).withColumnRenamed("h3", "h3_cell_id_orig")

print(f"Filtered to {carto_ma.count()} Massachusetts H3 cells with CARTO data")

In [ ]:
# MAGIC %md
# MAGIC ## Derive Urbanicity and K-Ring Size

In [ ]:
# Add urbanicity-based K-ring size
if urbanicity_col:
    # Use existing urbanicity column
    carto_with_kring = carto_ma.withColumn(
        "kring_size",
        when(col(urbanicity_col).isin("Very_High_density_urban", "High_density_urban", "very_high", "high"), 2)
        .when(col(urbanicity_col).isin("Medium_density_urban", "Low_density_urban", "medium", "low"), 3)
        .when(col(urbanicity_col).isin("Rural", "rural"), 8)
        .otherwise(DEFAULT_KRING)
    ).withColumn("urbanicity", col(urbanicity_col))
else:
    # Derive urbanicity from population density
    # Using population column and H3 cell area (~0.737 km² at res 8)
    H3_AREA_KM2 = 0.737
    carto_with_kring = carto_ma.withColumn(
        "pop_density", F.coalesce(col("population"), lit(0)) / H3_AREA_KM2
    ).withColumn(
        "urbanicity",
        when(col("pop_density") > 10000, "Very_High_density_urban")
        .when(col("pop_density") > 5000, "High_density_urban")
        .when(col("pop_density") > 1000, "Medium_density_urban")
        .when(col("pop_density") > 100, "Low_density_urban")
        .otherwise("Rural")
    ).withColumn(
        "kring_size",
        when(col("urbanicity").isin("Very_High_density_urban", "High_density_urban"), 2)
        .when(col("urbanicity").isin("Medium_density_urban", "Low_density_urban"), 3)
        .otherwise(8)
    )

print("Urbanicity distribution:")
display(carto_with_kring.groupBy("urbanicity", "kring_size").count().orderBy("kring_size"))

In [ ]:
# MAGIC %md
# MAGIC ## Calculate Trade Area Features Using K-Rings

In [ ]:
# Get demographic columns from config
demo_vars = config.get('carto_demographic_variables', {})
population_vars = demo_vars.get('population', [])

# Target demographic: Young adults 20-34 (fast, cheap pizza buyers)
young_adult_cols = [
    'male_20_to_24', 'female_20_to_24',
    'male_25_to_29', 'female_25_to_29',
    'male_30_to_34', 'female_30_to_34'
]

# CARTO POI columns
poi_cols = ['retail', 'education', 'financial', 'food_drink', 'healthcare', 'leisure', 'tourism', 'transportation']

# Filter to columns that exist
existing_young_adult_cols = [c for c in young_adult_cols if c in carto_with_kring.columns]
existing_poi_cols = [c for c in poi_cols if c in carto_with_kring.columns]

print(f"Young adult columns available: {existing_young_adult_cols}")
print(f"POI columns available: {existing_poi_cols}")

In [ ]:
# For each candidate H3 cell, aggregate features from its K-ring neighborhood
# Note: K-ring aggregation is computationally expensive, so we use SQL window functions

# First, add target demographic and total POI at cell level
target_demo_expr = " + ".join([f"COALESCE({c}, 0)" for c in existing_young_adult_cols]) if existing_young_adult_cols else "0"
total_poi_expr = " + ".join([f"COALESCE({c}, 0)" for c in existing_poi_cols]) if existing_poi_cols else "0"

candidates_base = carto_with_kring.withColumn(
    "target_demographic", expr(target_demo_expr)
).withColumn(
    "total_poi", expr(total_poi_expr)
).withColumn(
    "population", F.coalesce(col("population"), lit(0))
)

# For simplicity in this version, we use the cell-level values
# In production, you would use h3_kring() to aggregate neighbors
# This requires Databricks H3 SQL extensions

# Select candidate features
candidates = candidates_base.select(
    col("h3_cell_id"),
    col("urbanicity"),
    col("kring_size"),
    col("population"),
    col("target_demographic"),
    col("total_poi"),
    # Add H3 cell center as lat/lon for visualization
    expr("h3_centeraswkt(h3_cell_id)").alias("center_wkt")
)

# Parse center coordinates
candidates = candidates.withColumn(
    "latitude", expr("ST_Y(ST_GeomFromWKT(center_wkt, 4326))")
).withColumn(
    "longitude", expr("ST_X(ST_GeomFromWKT(center_wkt, 4326))")
).drop("center_wkt")

print(f"Generated {candidates.count()} candidate locations")
display(candidates.limit(10))

In [ ]:
# MAGIC %md
# MAGIC ## Filter to Top 25% Candidates

In [ ]:
# Create composite score for ranking
candidates_scored = candidates.withColumn(
    "composite_score",
    col("population") + col("target_demographic") * 10 + col("total_poi") * 5
)

# Calculate 75th percentile threshold
percentile_75 = candidates_scored.selectExpr(
    "percentile_approx(composite_score, 0.75) as p75"
).collect()[0]['p75']

print(f"75th percentile threshold: {percentile_75:,.0f}")

# Filter to top 25%
top_candidates = candidates_scored.filter(
    col("composite_score") >= percentile_75
)

top_count = top_candidates.count()
total_count = candidates_scored.count()
print(f"Top 25%: {top_count} candidates out of {total_count} total ({100*top_count/total_count:.1f}%)")

In [ ]:
# MAGIC %md
# MAGIC ## Write to Silver

In [ ]:
# Add processing timestamp
candidates_final = top_candidates.withColumn(
    "processing_timestamp", F.current_timestamp()
)

# Write to silver
(
    candidates_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(output_table)
)

print(f"Written {top_count} expansion candidates to {output_table}")

In [ ]:
# MAGIC %md
# MAGIC ## Summary by Urbanicity

In [ ]:
print("Expansion Candidates by Urbanicity:")
display(spark.sql(f"""
  SELECT
    urbanicity,
    kring_size,
    COUNT(*) as candidate_count,
    ROUND(AVG(population), 0) as avg_population,
    ROUND(AVG(target_demographic), 0) as avg_target_demo,
    ROUND(AVG(total_poi), 0) as avg_total_poi,
    ROUND(AVG(composite_score), 0) as avg_score
  FROM {output_table}
  GROUP BY urbanicity, kring_size
  ORDER BY kring_size, urbanicity
"""))